<div dir="rtl">

# סקירת נתונים ראשונית

מחברת קריאה בלבד. המטרה היא להכיר את 11 הטבלאות לפני שמתחילים לעבד אותן: מבנה, טיפוסים, ערכים חסרים, כפילויות ומבט על תחילת וסוף כל טבלה.

**המחברת הזו לא משנה שום נתון ולא כותבת שום קובץ.**

</div>

<div dir="rtl">

## ייבוא ספריות

</div>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

# show every column; long text is cut at 40 characters
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)

RAW = Path("..") / "data" / "raw"

print("pandas", pd.__version__)
print("numpy ", np.__version__)
print("raw data folder:", RAW.as_posix())

pandas 3.0.5
numpy  2.4.6
raw data folder: ../data/raw


<div dir="rtl">

## טעינת הטבלאות

כל טבלה נטענת למשתנה נפרד וגם למילון `tables`, כדי שיהיה נוח לעבור על כולן יחד.

</div>

In [2]:
tables = {}

orders = pd.read_csv(RAW / "olist_orders_dataset.csv", low_memory=False)
tables["orders"] = orders
order_items = pd.read_csv(RAW / "olist_order_items_dataset.csv", low_memory=False)
tables["order_items"] = order_items
order_payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv", low_memory=False)
tables["order_payments"] = order_payments
order_reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv", low_memory=False)
tables["order_reviews"] = order_reviews
customers = pd.read_csv(RAW / "olist_customers_dataset.csv", low_memory=False)
tables["customers"] = customers
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv", low_memory=False)
tables["sellers"] = sellers
products = pd.read_csv(RAW / "olist_products_dataset.csv", low_memory=False)
tables["products"] = products
geolocation = pd.read_csv(RAW / "olist_geolocation_dataset.csv", low_memory=False)
tables["geolocation"] = geolocation
category_translation = pd.read_csv(RAW / "product_category_name_translation.csv", low_memory=False)
tables["category_translation"] = category_translation
qualified_leads = pd.read_csv(RAW / "olist_marketing_qualified_leads_dataset.csv", low_memory=False)
tables["qualified_leads"] = qualified_leads
closed_deals = pd.read_csv(RAW / "olist_closed_deals_dataset.csv", low_memory=False)
tables["closed_deals"] = closed_deals

print(f"נטענו {len(tables)} טבלאות")

נטענו 11 טבלאות


<div dir="rtl">

## מבט על כלל הטבלאות

טבלה מסכמת אחת: כמה שורות ועמודות יש בכל טבלה, כמה תאים חסרים, כמה שורות כפולות מלאות וכמה זיכרון היא תופסת.

</div>

In [3]:
overview = pd.DataFrame([
    {
        "table":         name,
        "rows":          len(df),
        "cols":          df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "missing_%":     round(df.isna().sum().sum() / (len(df) * df.shape[1]) * 100, 2),
        "dup_rows":      int(df.duplicated().sum()),
        "memory_MB":     round(df.memory_usage(deep=True).sum() / 1024**2, 1),
    }
    for name, df in tables.items()
]).sort_values("rows", ascending=False).reset_index(drop=True)

display(overview)
print(f"total {overview['rows'].sum():,} rows across {len(overview)} tables")

,table,rows,cols,missing_cells,missing_%,dup_rows,memory_MB
0,geolocation,1000163,5,0,0.00,261831,50.1
1,order_items,112650,7,0,0.00,0,18.4
2,order_payments,103886,5,0,0.00,0,8.1
3,orders,99441,8,4908,0.62,0,21.9
4,customers,99441,5,0,0.00,0,11.0
5,order_reviews,99224,7,145903,21.01,0,17.8
6,products,32951,9,2448,0.83,0,3.7
7,qualified_leads,8000,4,60,0.19,0,0.9
8,sellers,3095,4,0,0.00,0,0.2
9,closed_deals,842,14,3300,27.99,0,0.3


total 1,559,764 rows across 11 tables


<div dir="rtl">

## פונקציית הסקירה

פונקציה אחת שמריצים על כל טבלה, כדי שהפלט יהיה אחיד וניתן להשוואה.

</div>

In [4]:
def review(df, name, n=10):
    """Read-only review of one table: shape, dtypes, nulls, uniques, duplicates, head and tail."""
    display(Markdown(f"### `{name}`: {len(df):,} rows x {df.shape[1]} cols"))

    profile = pd.DataFrame({
        "dtype":    df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "nulls":    df.isna().sum(),
        "null_%":   (df.isna().sum() / len(df) * 100).round(2),
        "unique":   df.nunique(dropna=True),
        "unique_%": (df.nunique(dropna=True) / len(df) * 100).round(2),
        "sample":   [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    display(profile)

    print(f"duplicate full rows    : {df.duplicated().sum():,}")

    pk = [c for c in df.columns if df[c].nunique(dropna=True) == len(df)]
    print(f"primary-key candidates : {pk if pk else 'none: no column is unique per row'}")

    for c in [c for c in df.columns if c.endswith("_id")]:
        u = df[c].nunique(dropna=True)
        if u != len(df):
            print(f"  {c}: {u:,} distinct of {len(df):,} rows -> repeats")

    display(Markdown(f"**first {n} rows**"))
    display(df.head(n))
    display(Markdown(f"**last {n} rows**"))
    display(df.tail(n))

<div dir="rtl">

---

# סקירה טבלה אחר טבלה

</div>

<div dir="rtl">

## orders

הזמנות: שורה לכל הזמנה, עם סטטוס וחמש חותמות זמן

</div>

In [5]:
review(orders, "orders")

### `orders`: 99,441 rows x 8 cols

,dtype,non_null,nulls,null_%,unique,unique_%,sample
order_id,str,99441,0,0.00,99441,100.00,e481f51cbdc54678b7cc49136f2d6af7
customer_id,str,99441,0,0.00,99441,100.00,9ef432eb6251297304e76186b10a928d
order_status,str,99441,0,0.00,8,0.01,delivered
order_purchase_timestamp,str,99441,0,0.00,98875,99.43,2017-10-02 10:56:33
order_approved_at,str,99281,160,0.16,90733,91.24,2017-10-02 11:07:15
order_delivered_carrier_date,str,97658,1783,1.79,81018,81.47,2017-10-04 19:55:00
order_delivered_customer_date,str,96476,2965,2.98,95664,96.20,2017-10-10 21:25:13
order_estimated_delivery_date,str,99441,0,0.00,459,0.46,2017-10-18 00:00:00


duplicate full rows    : 0
primary-key candidates : ['order_id', 'customer_id']


**first 10 rows**

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


**last 10 rows**

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
99431,b0f4af5c1b06e24fef510703bfe9f0a6,8e1ec396e317ff4c82a03ce16a0c3eb3,delivered,2017-10-27 15:21:00,2017-10-27 15:32:49,2017-10-30 15:44:34,2017-11-10 17:57:22,2017-11-22 00:00:00
99432,cfa78b997e329a5295b4ee6972c02979,a2f7428f0cafbc8e59f20e1444b67315,delivered,2017-12-20 09:52:41,2017-12-20 10:09:52,2017-12-20 20:25:25,2018-01-26 15:45:14,2018-01-18 00:00:00
99433,9115830be804184b91f5c00f6f49f92d,da2124f134f5dfbce9d06f29bdb6c308,delivered,2017-10-04 19:57:37,2017-10-04 20:07:14,2017-10-05 16:52:52,2017-10-20 20:25:45,2017-11-07 00:00:00
99434,aa04ef5214580b06b10e2a378300db44,f01a6bfcc730456317e4081fe0c9940e,delivered,2017-01-27 00:30:03,2017-01-27 01:05:25,2017-01-30 11:40:16,2017-02-07 13:15:25,2017-03-17 00:00:00
99435,880675dff2150932f1601e1c07eadeeb,47cd45a6ac7b9fb16537df2ccffeb5ac,delivered,2017-02-23 09:05:12,2017-02-23 09:15:11,2017-03-01 10:22:52,2017-03-06 11:08:08,2017-03-22 00:00:00
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00
99440,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-03-08 20:57:30,2018-03-09 11:20:28,2018-03-09 22:11:59,2018-03-16 13:08:30,2018-04-03 00:00:00


<div dir="rtl">

## order_items

פריטי הזמנה: הטבלה היחידה שמקשרת מוכר לכסף

</div>

In [6]:
review(order_items, "order_items")

### `order_items`: 112,650 rows x 7 cols

,dtype,non_null,nulls,null_%,unique,unique_%,sample
order_id,str,112650,0,0.0,98666,87.59,00010242fe8c5a6d1ba2dd792cb16214
order_item_id,int64,112650,0,0.0,21,0.02,1
product_id,str,112650,0,0.0,32951,29.25,4244733e06e7ecb4970a6e2683c13e61
seller_id,str,112650,0,0.0,3095,2.75,48436dade18ac8b2bce089ec2a041202
shipping_limit_date,str,112650,0,0.0,93318,82.84,2017-09-19 09:45:35
price,float64,112650,0,0.0,5968,5.30,58.9
freight_value,float64,112650,0,0.0,6999,6.21,13.29


duplicate full rows    : 0
primary-key candidates : none: no column is unique per row
  order_id: 98,666 distinct of 112,650 rows -> repeats
  order_item_id: 21 distinct of 112,650 rows -> repeats
  product_id: 32,951 distinct of 112,650 rows -> repeats
  seller_id: 3,095 distinct of 112,650 rows -> repeats


**first 10 rows**

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
6,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
7,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
8,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
9,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40


**last 10 rows**

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
112640,fffb9224b6fc7c43ebb0904318b10b5f,1,43423cdffde7fda63d0414ed38c11a73,b1fc4f64df5a0e8b6913ab38803c57a9,2017-11-03 02:55:58,55.00,34.19
112641,fffb9224b6fc7c43ebb0904318b10b5f,2,43423cdffde7fda63d0414ed38c11a73,b1fc4f64df5a0e8b6913ab38803c57a9,2017-11-03 02:55:58,55.00,34.19
112642,fffb9224b6fc7c43ebb0904318b10b5f,3,43423cdffde7fda63d0414ed38c11a73,b1fc4f64df5a0e8b6913ab38803c57a9,2017-11-03 02:55:58,55.00,34.19
112643,fffb9224b6fc7c43ebb0904318b10b5f,4,43423cdffde7fda63d0414ed38c11a73,b1fc4f64df5a0e8b6913ab38803c57a9,2017-11-03 02:55:58,55.00,34.19
112644,fffbee3b5462987e66fb49b1c5411df2,1,6f0169f259bb0ff432bfff7d829b9946,213b25e6f54661939f11710a6fddb871,2018-06-28 09:58:03,119.85,20.03
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72
112649,fffe41c64501cc87c801fd61db3f6244,1,350688d9dc1e75ff97be326363655e01,f7ccf836d21b2fb1de37564105216cc1,2018-06-12 17:10:13,43.00,12.79


<div dir="rtl">

## order_payments

תשלומים: שורה לכל תשלום בהזמנה

</div>

In [7]:
review(order_payments, "order_payments")

### `order_payments`: 103,886 rows x 5 cols

,dtype,non_null,nulls,null_%,unique,unique_%,sample
order_id,str,103886,0,0.0,99440,95.72,b81ef226f3fe1789b1e8b2acac839d17
payment_sequential,int64,103886,0,0.0,29,0.03,1
payment_type,str,103886,0,0.0,5,0.00,credit_card
payment_installments,int64,103886,0,0.0,24,0.02,8
payment_value,float64,103886,0,0.0,29077,27.99,99.33


duplicate full rows    : 0
primary-key candidates : none: no column is unique per row
  order_id: 99,440 distinct of 103,886 rows -> repeats


**first 10 rows**

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
5,298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
6,771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
7,3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
8,1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
9,0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


**last 10 rows**

,order_id,payment_sequential,payment_type,payment_installments,payment_value
103876,0872d0faafe0cb56e6f2c594cca1522c,1,credit_card,2,127.92
103877,3531718b24dfd5022fb0d101e10e33b9,1,credit_card,1,194.11
103878,2e3d96522c8c378d4135c12a65c977aa,1,credit_card,4,47.62
103879,c45067032fd84f4cf408730ff5205568,1,credit_card,2,198.94
103880,7159096c5aa9be77f7f0c26c01ee9793,1,credit_card,4,280.65
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54
103885,28bbae6599b09d39ca406b747b6632b1,1,boleto,1,191.58


<div dir="rtl">

## order_reviews

ביקורות: ציון לכל הזמנה, ואינו ייחודי

</div>

In [8]:
review(order_reviews, "order_reviews")

### `order_reviews`: 99,224 rows x 7 cols

,dtype,non_null,nulls,null_%,unique,unique_%,sample
review_id,str,99224,0,0.00,98410,99.18,7bc2406110b926393aa56f80a40eba40
order_id,str,99224,0,0.00,98673,99.44,73fc7af87114b39712e6da79b0a377eb
review_score,int64,99224,0,0.00,5,0.01,4
review_comment_title,str,11568,87656,88.34,4527,4.56,recomendo
review_comment_message,str,40977,58247,58.70,36159,36.44,Recebi bem antes do prazo estipulado.
review_creation_date,str,99224,0,0.00,636,0.64,2018-01-18 00:00:00
review_answer_timestamp,str,99224,0,0.00,98248,99.02,2018-01-18 21:46:59


duplicate full rows    : 0


primary-key candidates : none: no column is unique per row
  review_id: 98,410 distinct of 99,224 rows -> repeats
  order_id: 98,673 distinct of 99,224 rows -> repeats


**first 10 rows**

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comp...,2018-03-01 00:00:00,2018-03-02 10:26:53
5,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,NaN,NaN,2018-04-13 00:00:00,2018-04-16 00:39:37
6,07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,NaN,NaN,2017-07-16 00:00:00,2017-07-18 19:30:34
7,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,NaN,NaN,2018-08-14 00:00:00,2018-08-14 21:36:06
8,a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,NaN,NaN,2017-05-17 00:00:00,2017-05-18 12:05:37
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca ...,2018-05-22 00:00:00,2018-05-23 16:45:47


**last 10 rows**

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
99214,3c569245c7610d9a51eff50032194ea3,bf25ff262495b0b876bc4f594f736ecf,5,NaN,NaN,2018-05-23 00:00:00,2018-05-26 21:24:06
99215,a709d176f59bc3af77f4149c96bae357,d5cb12269711bd1eaf7eed8fd32a7c95,3,NaN,"O produto não foi enviado com NF, nã...",2018-05-19 00:00:00,2018-05-20 21:51:06
99216,115d72425fc59eec9a2e9d7594c6d93d,acd45245723df7cb52772a34416b41b1,3,NaN,NaN,2018-05-08 00:00:00,2018-05-10 14:37:03
99217,c6b270c61f67c9f7cb07d84ea8aeaf8b,48f7ee67313eda32bfcf5b9c1dd9522d,5,NaN,NaN,2017-12-13 00:00:00,2017-12-14 11:09:36
99218,af2dc0519de6e0720ef0c74292fb4114,d699c734a0b1c8111f2272a3f36d398c,5,NaN,NaN,2018-04-27 00:00:00,2018-04-30 01:18:57
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super ráp...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13
99223,efe49f1d6f951dd88b51e6ccd4cc548f,90531360ecb1eec2a1fbb265a0db0508,1,NaN,meu produto chegou e ja tenho que de...,2017-07-03 00:00:00,2017-07-03 21:01:49


<div dir="rtl">

## customers

לקוחות: שורה לכל הזמנה ולא לכל אדם

</div>

In [9]:
review(customers, "customers")

### `customers`: 99,441 rows x 5 cols

,dtype,non_null,nulls,null_%,unique,unique_%,sample
customer_id,str,99441,0,0.0,99441,100.00,06b8999e2fba1a1fbc88172c00ba8bc7
customer_unique_id,str,99441,0,0.0,96096,96.64,861eff4711a542e4b93843c6dd7febb0
customer_zip_code_prefix,int64,99441,0,0.0,14994,15.08,14409
customer_city,str,99441,0,0.0,4119,4.14,franca
customer_state,str,99441,0,0.0,27,0.03,SP


duplicate full rows    : 0
primary-key candidates : ['customer_id']
  customer_unique_id: 96,096 distinct of 99,441 rows -> repeats


**first 10 rows**

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
6,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
7,5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
9,4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


**last 10 rows**

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
99431,be842c57a8c5a62e9585dd72f22b6338,ca186065de6e2d01cfc99763e6d62048,99150,marau,RS
99432,f255d679c7c86c24ef4861320d5b7675,d111b06b6f3a2add0d2241325f65b5ca,13500,rio claro,SP
99433,14308d2303a3e2bdf4939b86c46d2679,e7f8760e2bbd2f1986bebd99596c088e,66033,belem,PA
99434,f5a0b560f9e9427792a88bec97710212,b3e53d18a997f27a3ffd16da497eaf58,7790,cajamar,SP
99435,7fe2e80252a9ea476f950ae8f85b0f8f,4b5820135d360a45552b5163835b1d89,35500,divinopolis,MG
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS
99440,274fa6071e5e17fe303b9748641082c8,84732c5050c01db9b23e19ba39899398,6703,cotia,SP


<div dir="rtl">

## sellers

מוכרים: רישום של מוכרים שבאמת מכרו

</div>

In [10]:
review(sellers, "sellers")

### `sellers`: 3,095 rows x 4 cols

,dtype,non_null,nulls,null_%,unique,unique_%,sample
seller_id,str,3095,0,0.0,3095,100.00,3442f8959a84dea7ee197c632cb2df15
seller_zip_code_prefix,int64,3095,0,0.0,2246,72.57,13023
seller_city,str,3095,0,0.0,611,19.74,campinas
seller_state,str,3095,0,0.0,23,0.74,SP


duplicate full rows    : 0
primary-key candidates : ['seller_id']


**first 10 rows**

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
5,c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
6,e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
7,1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
8,768a86e36ad6aae3d03ee3c6433d61df,1529,sao paulo,SP
9,ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


**last 10 rows**

,seller_id,seller_zip_code_prefix,seller_city,seller_state
3085,9d489893ff6af05029671b785f54b998,13613,leme,SP
3086,725af026f869236a8dd0ceb98564d156,35530,claudio,MG
3087,f00f5b35d0abcacbdd863672f4bb2c1a,1238,sao paulo,SP
3088,7bac63f6603d382cc8d0832eb6c100a8,5713,sao paulo,SP
3089,f1fdf2d13186575751aa25876536d85c,5314,sao paulo,SP
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS
3094,9e25199f6ef7e7c347120ff175652c3b,12051,taubate,SP


<div dir="rtl">

## products

מוצרים: קטלוג עם קטגוריה ומידות

</div>

In [11]:
review(products, "products")

### `products`: 32,951 rows x 9 cols

,dtype,non_null,nulls,null_%,unique,unique_%,sample
product_id,str,32951,0,0.00,32951,100.00,1e9e8ef04dbcff4541ed26657ea517e5
product_category_name,str,32341,610,1.85,73,0.22,perfumaria
product_name_lenght,float64,32341,610,1.85,66,0.20,40.0
product_description_lenght,float64,32341,610,1.85,2960,8.98,287.0
product_photos_qty,float64,32341,610,1.85,19,0.06,1.0
product_weight_g,float64,32949,2,0.01,2204,6.69,225.0
product_length_cm,float64,32949,2,0.01,99,0.30,16.0
product_height_cm,float64,32949,2,0.01,102,0.31,10.0
product_width_cm,float64,32949,2,0.01,95,0.29,14.0


duplicate full rows    : 0
primary-key candidates : ['product_id']


**first 10 rows**

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60.0,745.0,1.0,200.0,38.0,5.0,11.0
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,56.0,1272.0,4.0,18350.0,70.0,24.0,44.0
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56.0,184.0,2.0,900.0,40.0,8.0,40.0
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,57.0,163.0,1.0,400.0,27.0,13.0,17.0
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,36.0,1156.0,1.0,600.0,17.0,10.0,12.0


**last 10 rows**

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
32941,6ec96c91757fad0aecafc0ee7f262dcc,bebes,62.0,1417.0,1.0,9550.0,36.0,35.0,35.0
32942,16280ca280a86fee2ba3c928ed04439f,moveis_decoracao,64.0,236.0,11.0,2200.0,31.0,11.0,26.0
32943,3becff10d1deb92b02f2a1ee62a04524,informatica_acessorios,54.0,1520.0,2.0,6150.0,30.0,30.0,20.0
32944,1a14237ecc2fe3772b55c8d4e11ccb35,moveis_decoracao,58.0,1405.0,3.0,150.0,35.0,2.0,25.0
32945,c4e71b64511b959455e2107fe7859020,utilidades_domesticas,59.0,1371.0,2.0,200.0,18.0,15.0,15.0
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0
32950,106392145fca363410d287a815be6de4,cama_mesa_banho,58.0,309.0,1.0,2083.0,12.0,2.0,7.0


<div dir="rtl">

## geolocation

גאולוקציה: קידומת מיקוד לקואורדינטות

</div>

In [12]:
review(geolocation, "geolocation")

### `geolocation`: 1,000,163 rows x 5 cols

,dtype,non_null,nulls,null_%,unique,unique_%,sample
geolocation_zip_code_prefix,int64,1000163,0,0.0,19015,1.90,1037
geolocation_lat,float64,1000163,0,0.0,717363,71.72,-23.545621
geolocation_lng,float64,1000163,0,0.0,717615,71.75,-46.639292
geolocation_city,str,1000163,0,0.0,8011,0.80,sao paulo
geolocation_state,str,1000163,0,0.0,27,0.00,SP


duplicate full rows    : 261,831
primary-key candidates : none: no column is unique per row


**first 10 rows**

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
5,1012,-23.547762,-46.635361,são paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP


**last 10 rows**

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1000153,99970,-28.343273,-51.873734,ciriaco,RS
1000154,99950,-28.070493,-52.011342,tapejara,RS
1000155,99965,-28.180655,-52.034367,agua santa,RS
1000156,99950,-28.072188,-52.011272,tapejara,RS
1000157,99950,-28.068864,-52.012964,tapejara,RS
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS
1000162,99950,-28.070104,-52.018658,tapejara,RS


<div dir="rtl">

## category_translation

תרגום קטגוריות מפורטוגזית לאנגלית

</div>

In [13]:
review(category_translation, "category_translation")

### `category_translation`: 71 rows x 2 cols

,dtype,non_null,nulls,null_%,unique,unique_%,sample
product_category_name,str,71,0,0.0,71,100.0,beleza_saude
product_category_name_english,str,71,0,0.0,71,100.0,health_beauty


duplicate full rows    : 0
primary-key candidates : ['product_category_name', 'product_category_name_english']


**first 10 rows**

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


**last 10 rows**

,product_category_name,product_category_name_english
61,musica,music
62,casa_conforto_2,home_comfort_2
63,portateis_casa_forno_e_cafe,small_appliances_home_oven_and_coffee
64,cds_dvds_musicais,cds_dvds_musicals
65,dvds_blu_ray,dvds_blu_ray
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes
70,seguros_e_servicos,security_and_services


<div dir="rtl">

## qualified_leads

לידים מוכשרים: נקודת המוצא של המשפך

</div>

In [14]:
review(qualified_leads, "qualified_leads")

### `qualified_leads`: 8,000 rows x 4 cols

,dtype,non_null,nulls,null_%,unique,unique_%,sample
mql_id,str,8000,0,0.00,8000,100.00,dac32acd4db4c29c230538b72f8dd87d
first_contact_date,str,8000,0,0.00,336,4.20,2018-02-01
landing_page_id,str,8000,0,0.00,495,6.19,88740e65d5d6b056e0cda098e1ea6313
origin,str,7940,60,0.75,10,0.12,social


duplicate full rows    : 0
primary-key candidates : ['mql_id']
  landing_page_id: 495 distinct of 8,000 rows -> repeats


**first 10 rows**

,mql_id,first_contact_date,landing_page_id,origin
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search
5,28bdfd5f057764b54c38770f95c69f2f,2018-01-14,22c29808c4f815213303f8933030604c,organic_search
6,126a0d10becbaafcb2e72ce6848cf32c,2018-05-15,6a110e795dd487f1cf8d7583671987af,email
7,f76136f54d14a3345951f25b7932366b,2018-05-24,d51b0d02f063ba1d053db6d97226eec3,email
8,2f838cade4a6012a6cb1016d1d8d95ed,2017-11-10,aeac92c0f5ae22a04ed3b746cce3a1b6,organic_search
9,7281942387a1a0c3f72a50a8b0bb0920,2017-12-25,88740e65d5d6b056e0cda098e1ea6313,social


**last 10 rows**

,mql_id,first_contact_date,landing_page_id,origin
7990,3039a72740ccf74a6c840aacbef79b0b,2018-02-20,54d5e01ab35155d4bd8cf723fc6f04a0,email
7991,2e57665b8faf05c967a801eb5aedfa0a,2018-05-25,40dec9f3d5259a3d2dbcdab2114fae47,paid_search
7992,c24f9ae141fa02c7fa1deea7e1149557,2018-01-07,88740e65d5d6b056e0cda098e1ea6313,social
7993,aedd87de3760230b3c1e74e37b875a38,2017-08-15,92bd5f77d493a36dac9bddb66d1b9dbf,organic_search
7994,40955be51cc85c5a5a6cdd12b19e9f10,2018-02-04,88740e65d5d6b056e0cda098e1ea6313,social
7995,feaba3ffcd2ff97501696c7f9a42f41c,2018-05-22,e42a14209c69c3e9cc6b042620465f12,paid_search
7996,a79cb53cd009ab92e0143b92baa2407b,2018-03-27,c494978688ccf66ad9fad3d6a3338c22,paid_search
7997,68f049a23ab109c6a0f6989bb9a02994,2017-08-27,b48ec5f3b04e9068441002a19df93c6c,organic_search
7998,4f8c96e2509b984329044c6682c88ee9,2017-10-06,a56671a54260a44923d32c2f08fad39c,organic_search
7999,8a6492305a5fbcdcdd1a7f5a90764c07,2018-01-11,b76ef37428e6799c421989521c0e5077,direct_traffic


<div dir="rtl">

## closed_deals

עסקאות שנסגרו: טבלת הגשר בין המשפך למוכרים

</div>

In [15]:
review(closed_deals, "closed_deals")

### `closed_deals`: 842 rows x 14 cols

,dtype,non_null,nulls,null_%,unique,unique_%,sample
mql_id,str,842,0,0.00,842,100.00,5420aad7fec3549a85876ba1c529bd84
seller_id,str,842,0,0.00,842,100.00,2c43fb513632d29b3b58df74816f1b06
sdr_id,str,842,0,0.00,32,3.80,a8387c01a09e99ce014107505b92388c
sr_id,str,842,0,0.00,22,2.61,4ef15afb4b2723d8f3d81e51ec7afefe
won_date,str,842,0,0.00,824,97.86,2018-02-26 19:58:54
business_segment,str,841,1,0.12,33,3.92,pet
lead_type,str,836,6,0.71,8,0.95,online_medium
lead_behaviour_profile,str,665,177,21.02,9,1.07,cat
has_company,object,63,779,92.52,2,0.24,True
has_gtin,object,64,778,92.40,2,0.24,True


duplicate full rows    : 0
primary-key candidates : ['mql_id', 'seller_id']
  sdr_id: 32 distinct of 842 rows -> repeats
  sr_id: 22 distinct of 842 rows -> repeats


**first 10 rows**

,mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue
0,5420aad7fec3549a85876ba1c529bd84,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0
1,a555fb36b9368110ede0f043dfc3b9a0,bbb7d7893a450660432ea6652310ebb7,09285259593c61296eef10c734121d5b,d3d1e91a157ea7f90548eef82f1955e3,2018-05-08 20:17:59,car_accessories,industry,eagle,NaN,NaN,NaN,reseller,NaN,0.0
2,327174d3648a2d047e8940d7d15204ca,612170e34b97004b3ba37eae81836b4c,b90f87164b5f8c2cfa5c8572834dbe3f,6565aa9ce3178a5caf6171827af3a9ba,2018-06-05 17:27:23,home_appliances,online_big,cat,NaN,NaN,NaN,reseller,NaN,0.0
3,f5fee8f7da74f4887f5bcae2bafb6dd6,21e1781e36faf92725dde4730a88ca0f,56bf83c4bb35763a51c2baab501b4c67,d3d1e91a157ea7f90548eef82f1955e3,2018-01-17 13:51:03,food_drink,online_small,NaN,NaN,NaN,NaN,reseller,NaN,0.0
4,ffe640179b554e295c167a2f6be528e0,ed8cb7b190ceb6067227478e48cf8dde,4b339f9567d060bcea4f5136b9f5949e,d3d1e91a157ea7f90548eef82f1955e3,2018-07-03 20:17:45,home_appliances,industry,wolf,NaN,NaN,NaN,manufacturer,NaN,0.0
5,b94fba7670eeb44dce2a0d8eb790e9f5,1c742ac33582852aaf3bcfbf5893abcf,fdb16d3cbbeb5798f2f66c4096be026d,495d4e95a8cf8bbf8b432b612a2aa328,2018-02-07 18:04:05,health_beauty,online_medium,NaN,NaN,NaN,NaN,manufacturer,NaN,0.0
6,c3e30ed7ac989117c7e1e719b4ac128f,92d7568ad0c5c76fd7d341b2d46f24d6,4b339f9567d060bcea4f5136b9f5949e,85fc447d336637ba1df43e793199fbc8,2018-04-16 18:18:22,computers,online_medium,wolf,NaN,NaN,NaN,reseller,NaN,0.0
7,b02c89251106e1fdd9d92744be9f94f2,44ed138eca6214d572ce1d813fb0049b,34d40cdaf94010a1d05b0d6212f9e909,4ef15afb4b2723d8f3d81e51ec7afefe,2018-04-17 17:01:57,health_beauty,offline,cat,NaN,NaN,NaN,manufacturer,NaN,0.0
8,a90a37898cc5f2718385a2fb981caaff,0b28859cd04d23edefee9c591fb03cd8,f42a2bd194f7802ab052a815c8de65b7,6565aa9ce3178a5caf6171827af3a9ba,2018-05-14 18:37:15,household_utilities,offline,eagle,NaN,NaN,NaN,reseller,NaN,0.0
9,0173e8d8b1d94a355b440fb67388f532,87d73636a3acf123e842bb890a4db036,9d12ef1a7eca3ec58c545c678af7869c,9e4d1098a3b0f5da39b0bc48f9876645,2018-04-24 03:00:00,household_utilities,online_medium,eagle,NaN,NaN,NaN,manufacturer,NaN,0.0


**last 10 rows**

,mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue
832,f8e1b8b3a3fe9b27a64dfcaaa8fc6054,b6392ab5fdb153fb3c6c7c0e174cdc29,4b339f9567d060bcea4f5136b9f5949e,4ef15afb4b2723d8f3d81e51ec7afefe,2018-04-04 20:33:19,audio_video_electronics,online_medium,NaN,NaN,NaN,NaN,reseller,NaN,0.0
833,cd803c8ab3c2afa5430eeb195bcebe6a,33dd941c27854f7625b968cc6195a552,a8387c01a09e99ce014107505b92388c,85fc447d336637ba1df43e793199fbc8,2018-03-28 22:42:55,household_utilities,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0
834,676207ddcc8087437135a4b461fc07cb,c70a353f02429c00775a46a75fb787da,370c9f455f93a9a96cbe9bea48e70033,4ef15afb4b2723d8f3d81e51ec7afefe,2018-04-20 12:27:44,health_beauty,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0
835,98b18a14a0472c9e4a4eb0bb77ae3592,4fe2f67634d00c021aa01f96b6f2d68e,9d12ef1a7eca3ec58c545c678af7869c,a8387c01a09e99ce014107505b92388c,2018-06-07 21:23:45,food_supplement,online_small,cat,NaN,NaN,NaN,reseller,NaN,0.0
836,e7f70a2ff45e44c7399bebacf3e4771d,b21768cfefb49b94e9bcd38037e93d1f,56bf83c4bb35763a51c2baab501b4c67,c638112b43f1d1b86dcabb0da720c901,2018-02-09 12:16:16,car_accessories,online_medium,NaN,NaN,NaN,NaN,manufacturer,NaN,0.0
837,df9a2abe2ad3f2d8758b675ac8331ecf,bfcc27719640628da877db48b672b169,45749fb708130f78d0db07d8d80f030b,a8387c01a09e99ce014107505b92388c,2018-08-01 20:22:31,home_decor,offline,cat,NaN,NaN,NaN,reseller,NaN,0.0
838,8723201570415eb23b4f095d8c20a27c,4a82eab98441aeb64566e2776c1fb2b6,370c9f455f93a9a96cbe9bea48e70033,4b339f9567d060bcea4f5136b9f5949e,2018-09-27 18:58:41,construction_tools_house_garden,online_big,wolf,NaN,NaN,NaN,reseller,132.0,200000.0
839,2e57665b8faf05c967a801eb5aedfa0a,55031883943971ca22db6894574cfe2c,370c9f455f93a9a96cbe9bea48e70033,a8387c01a09e99ce014107505b92388c,2018-06-06 20:15:26,computers,online_medium,wolf,NaN,NaN,NaN,reseller,NaN,0.0
840,40955be51cc85c5a5a6cdd12b19e9f10,c0e933c238e41f0cd459d6025ee9b364,068066e24f0c643eb1d089c7dd20cd73,d3d1e91a157ea7f90548eef82f1955e3,2018-03-06 15:52:23,pet,online_beginner,cat,NaN,NaN,NaN,reseller,NaN,0.0
841,8a6492305a5fbcdcdd1a7f5a90764c07,1bba9c94eb5c4e82fba9f7286fe81025,56bf83c4bb35763a51c2baab501b4c67,de63de0d10a6012430098db33c679b0b,2018-05-02 13:31:32,household_utilities,industry,eagle,NaN,NaN,NaN,manufacturer,NaN,0.0


<div dir="rtl">

---

## בדיקות ממוקדות

התא הבא מחשב מהנתונים הגולמיים את המספרים שהסיכום ופרק המתודולוגיה בספר מצטטים: ריבוי הביקורות להזמנה, הזמנות עם כמה מוכרים, ציון הביקורת לפי סטטוס ההזמנה, היום שבו נתוני ההזמנות נגמרים בפועל, וטווח התאריכים של הלידים.

</div>

In [16]:
# Each figure below is quoted in the summary or in the book's methodology chapter.
orders["order_purchase_timestamp"] = pd.to_datetime(orders.order_purchase_timestamp)
rv = order_reviews.assign(review_creation_date=pd.to_datetime(order_reviews.review_creation_date),
                          review_answer_timestamp=pd.to_datetime(order_reviews.review_answer_timestamp))

# 1. several reviews on one order, and the tie that the creation date alone cannot break
per_order = rv.groupby("order_id")
multi_ids = per_order.size()[lambda s: s > 1].index
multi = rv[rv.order_id.isin(multi_ids)]
sellers_per_order = order_items.groupby("order_id").seller_id.nunique()
multi_with_items = sellers_per_order.reindex(multi_ids).dropna()
print("orders with more than one review       :", len(multi_ids), "orders,", len(multi), "rows")
print("  of them, with differing scores        :", int((multi.groupby("order_id").review_score.nunique() > 1).sum()))
print("  of them, all reviews on the same date :", int((multi.groupby("order_id").review_creation_date.nunique() == 1).sum()))
print("  of them, with items / single seller   : %d / %d (%.1f%%)"
      % (len(multi_with_items), int((multi_with_items == 1).sum()), 100 * (multi_with_items == 1).mean()))
print("orders with items of several sellers   : %d (up to %d sellers)" % (int((sellers_per_order > 1).sum()), sellers_per_order.max()))

# 2. mean review score by order status, one review per order (the earliest), over the whole dataset
first_review = rv.sort_values(["order_id", "review_creation_date", "review_answer_timestamp"]).drop_duplicates("order_id")
by_status = (first_review.merge(orders[["order_id", "order_status"]], on="order_id", validate="one_to_one")
             .groupby("order_status").review_score.agg(["size", "mean"]).round(2).sort_values("size", ascending=False))
print("\norders not delivered:", int((orders.order_status != "delivered").sum()))
display(by_status)

# 3. where the order data really ends
daily = orders.groupby(orders.order_purchase_timestamp.dt.normalize()).size()
data_end = daily[daily >= 10].index.max()
after = orders[orders.order_purchase_timestamp.dt.normalize() > data_end]
print("last day with at least ten orders      :", data_end.date(), "with", daily[data_end], "orders")
print("orders after it                        : %d, of them canceled: %d, last on %s"
      % (len(after), int((after.order_status == "canceled").sum()), after.order_purchase_timestamp.max().date()))

# 4. the lead window, against the official description (June 2017 to June 2018)
contact = pd.to_datetime(qualified_leads.first_contact_date)
print("first contact of the leads             :", contact.min().date(), "to", contact.max().date())

orders with more than one review       : 547 orders, 1098 rows
  of them, with differing scores        : 202
  of them, all reviews on the same date : 155
  of them, with items / single seller   : 544 / 531 (97.6%)
orders with items of several sellers   : 1278 (up to 5 sellers)



orders not delivered: 2963


,size,mean
order_status,,
delivered,95832,4.16
shipped,1032,2.01
canceled,605,1.80
unavailable,595,1.53
invoiced,309,1.66
processing,295,1.28
created,3,2.33
approved,2,2.50


last day with at least ten orders      : 2018-08-29 with 14 orders
orders after it                        : 25, of them canceled: 24, last on 2018-10-17
first contact of the leads             : 2017-06-14 to 2018-05-31


<div dir="rtl">

---

## סיכום הסקירה

הממצאים כאן נבדקו מול הנתונים הגולמיים, ולכל אחד נבדק גם מקור הבעיה. התאים שלמעלה מציגים את הפרופיל של כל טבלה, ותא הבדיקות הממוקדות מחשב את המספרים מהנתונים הגולמיים שהספר מצטט. שאר הבדיקות נערכו בשלב הסקירה. הסקירה אינה משנה שום נתון.

**כל בעיה שדרשה הכרעה כבר הוכרעה, וההכרעה מופיעה מתחתיה בהדגשה. ריכוז של כל הכללים נמצא בסעיף ז בסוף המחברת.**

### א. בעיות שחייבים לטפל בהן לפני חיבור טבלאות, וההכרעה לכל אחת

**1. המפתחות בטבלת הביקורות אינם ייחודיים, ומדובר בשתי תופעות שונות**

| תופעה | היקף | מה קורה בפועל |
|---|---|---|
| אותו `review_id` על כמה הזמנות | 789 מזהים, 1,603 שורות | בכל 789 המקרים הציון ותאריך היצירה זהים. מדובר בביקורת אחת שמקושרת לכמה הזמנות של אותה רכישה |
| אותה הזמנה עם כמה ביקורות | 547 הזמנות, 1,098 שורות | לכל אחת מזהה ביקורת שונה, וב-202 מהן גם **הציון שונה** |

אין בטבלה אף שורה כפולה מלאה, ולכן `drop_duplicates` לבדו אינו פותר את הבעיה. חיבור בלי הכרעה מפורשת מנפח שורות ומכפיל הכנסות בלי שום אזהרה.

> **ההכרעה:** נבחרה הביקורת הראשונה לפי תאריך, ושבירת השוויון נעשית לפי `review_answer_timestamp` שהוא ייחודי בכל המקרים. בנוסף, כל חיבור של ביקורות מחייב צמצום מוקדם לשורה אחת לכל הזמנה.

**2. טבלת הגאולוקציה אינה טבלת חיפוש אלא רשימת נקודות**

מיליון שורות מול 19,015 קידומות מיקוד בלבד, בממוצע 52.6 שורות לקידומת ועד 1,146 במקרה הקיצוני. יש בה 261,831 שורות זהות לחלוטין, כי כתובות רבות נופלות על אותן קואורדינטות מעוגלות. חיבור ישיר מכפיל את מספר השורות.

> **ההכרעה:** כל שימוש בטבלה מחייב צמצום מוקדם לשורה אחת לכל קידומת מיקוד, למשל ממוצע קואורדינטות.

**3. סכום התשלום אינו ניתן לייחוס למוכר**

1,278 הזמנות מכילות פריטים של יותר ממוכר אחד, ועד חמישה מוכרים בהזמנה אחת. לכן `payment_value` הוא ברמת ההזמנה ואי אפשר לפצל אותו בין מוכרים.

> **ההכרעה:** כל חישוב הכנסות למוכר מתבסס על `order_items.price` בלבד.

### ב. בעיות שמשפיעות על המספרים

| נושא | היקף | מה זה אומר | **ההכרעה** |
|---|---|---|---|
| הזמנות שלא סופקו | 2,963 הזמנות, 2,453 שורות פריט | סינון לסטטוס שסופק מוריד כ-3 אחוזים מהנתונים | **סינון להזמנות שסופקו בלבד, להכנסות, לביקורות ולמשך הפעילות.** בכלל הנתונים ציון הביקורת הממוצע הוא 4.16 בהזמנות שסופקו, ובין 1.28 ל-2.01 בכל סטטוס אחר שיש בו יותר ממאה ביקורות |
| עסקאות בלי מוכר בטבלת המוכרים | 462 מתוך 842 | אינה תקלה. כל 3,095 המוכרים בטבלה מופיעים בטבלת הפריטים, ולכן הטבלה מכילה רק מוכרים שמכרו בפועל. אף אחד מ-462 אלה אינו מופיע בטבלת הפריטים, כלומר הם חתמו ולא מכרו עד סוף הנתונים | **שיעור ההפעלה והערך למוכר מדווחים כשתי תוצאות נפרדות.** 462 אינם נספרים כציון אפס, משום שספירה כזו מענישה עסקאות שלא היה להן זמן למכור: 177 עסקאות נחתמו פחות מתשעים יום לפני סוף הנתונים או אחריו, 53 מהן אחריו, כפי שמחשבות מחברת 02 ונספח ו במחברת 03 |
| ערוץ מקור חסר מול ערוץ לא ידוע | 60 ערכים ריקים מול 1,099 בערך `unknown` | שני דברים שונים לגמרי. הערך `unknown` הוא קטגוריה אמיתית בנתוני המקור ולא ערך חסר | **הערך `unknown` מוצג בכל טבלה תיאורית אך מוחרג מכל דירוג והמלצה**, משום שאינו ערוץ שאפשר להגדיל בו את ההשקעה, ומשום שמוכר אחד אחראי לכמחצית מההכנסות שלו. ארבעת המוכרים שמגיעים מהערכים הריקים מוצאים מכל השוואה בין ערוצים |
| שדות מוצהרים בטבלת העסקאות | `has_company` ריק ב-92.5 אחוז, `declared_monthly_revenue` שווה אפס ב-797 מתוך 842 | אלה הצהרות של הליד בזמן ההרשמה ולא מדדים | **אין שימוש בשדות האלה בשום שלב של הניתוח** |

### ג. סתירות בחותמות הזמן

| בדיקה | תוצאה |
|---|---|
| הזמנה שסופקה לפני שנרכשה | 0 |
| אישור לפני רכישה | 0 |
| מסירה לשליח לפני האישור | **1,359** |
| הגעה ללקוח לפני המסירה לשליח | **23** |
| סטטוס "סופק" בלי תאריך אספקה | **8** |
| אספקה מאוחרת מהצפי | 7,827 (איחור אמיתי, לא תקלה) |

1,359 ו-23 ו-8 הם חוסר עקביות אמיתי בנתוני המקור. הם קטנים בהיקפם, אבל אם מחשבים משך אספקה הם יפיקו ערכים שליליים.

> **ההכרעה:** השורות האלה מסוננות בכל חישוב שנוגע למשך אספקה. הן אינן משפיעות על ההכנסות או על ציון הערך.

### ד. נקודות שכדאי להכיר

* **610 מוצרים בלי שום מטא-נתונים.** אותן 610 שורות חסרות גם קטגוריה, גם אורך שם, גם אורך תיאור וגם מספר תמונות. זו קבוצה אחת ולא ארבע בעיות נפרדות.
* **ארבעה מוצרים במשקל אפס.** רלוונטי רק אם משתמשים במידות.
* **שתי קטגוריות בלי תרגום לאנגלית**, `pc_gamer` ו-`portateis_cozinha_e_preparadores_de_alimentos`, בהיקף זניח: 13 מוצרים ו-24 שורות פריט בלבד. חיבור מסוג left join שומר אותן.
* **`customer_id` נוצר מחדש בכל הזמנה.** יש 99,441 מזהים מול 96,096 אנשים אמיתיים. 2,997 אנשים ביצעו יותר מהזמנה אחת, והשיא הוא 17 הזמנות לאדם אחד. לספירת אנשים חייבים להשתמש ב-`customer_unique_id` ולא במזהה ההזמנה.
* **42 נקודות ציון מחוץ לגבולות ברזיל**, ו-8,011 כתיבים שונים של שמות ערים. רלוונטי רק לניתוח גיאוגרפי.
* **רווחים מיותרים בתחילת הטקסט ובסופו**: 11,449 ערכים בשדות תגובת הביקורת, ובנוסף ערך אחד בשם עיר בטבלת הגאולוקציה, כלומר 11,450 בסך הכול.

### ה. בעיות בשמות

| סוג | פירוט |
|---|---|
| שגיאת כתיב במקור | `product_name_lenght` וגם `product_description_lenght`, במקום `length`, ואסור לתקן אותן, כי כך הן מופיעות בקובץ |
| אותו מושג בשלושה שמות | קידומת המיקוד נקראת `customer_zip_code_prefix`, `seller_zip_code_prefix` ו-`geolocation_zip_code_prefix`, וצריך שם אחיד לפני חיבור על מיקום |
| חריגה בשמות הקבצים | עשרה קבצים בתבנית `olist_*_dataset.csv`, וקובץ התרגום יוצא דופן בלי התחילית <code dir="ltr">olist_</code> ובלי החלק <code dir="ltr">_dataset</code> |

### ו. מה נבדק ונמצא תקין

חשוב לא פחות מהבעיות. הבדיקות הבאות עברו בלי ממצא:

* **שלמות קשרים מלאה.** אפס מפתחות יתומים בכל החיבורים שנבדקו: פריטים להזמנות, פריטים למוכרים, פריטים למוצרים, הזמנות ללקוחות, תשלומים להזמנות, ביקורות להזמנות ועסקאות ללידים.
* **כל המפתחות הראשיים ייחודיים** בטבלאות ההזמנות, הלקוחות, המוכרים, המוצרים והלידים.
* **אפס שורות כפולות מלאות** בכל הטבלאות למעט הגאולוקציה.
* **כל עשר עמודות התאריך מומרות לתאריך בלי שגיאה.**
* **אין ערכים בלתי אפשריים**: אין מחיר שלילי או אפס, אין משלוח שלילי, אין תשלום שלילי, ואין ציון ביקורת מחוץ לטווח המותר של 1 עד 5 כוכבים.
* **סכומי הכסף מתיישבים**: 98,089 מתוך 98,665 הזמנות מתאימות בדיוק בין סכום הפריטים לסכום התשלומים, ורק 249 חורגות ביותר מריאל אחד.
* **מספרי השורות אומתו מול הקבצים עצמם** בכלי חיצוני שאינו pandas, וכל 11 הטבלאות תאמו.

### ז. ההחלטות שהתקבלו וחלות על ההמשך

שלוש השאלות שעלו מהסקירה הוכרעו, יחד עם שש עשרה נוספות. תשע עשרה ההחלטות האלה מתורגמות לעשרים וארבעה הכללים שבטבלה, משום שהחלטה אחת מייצרת לעיתים יותר מכלל תפעולי אחד. להלן הכללים שכל מחברת בהמשך מחויבת להם. **הנימוק המלא לכל כלל, החלופות שנבחנו ומה נדחה, נמצאים בפרק המתודולוגיה שבקובץ `book/book.md` שבתיקיית הספר.** הטבלה כאן היא תזכורת בלבד ואינה המקור הקובע.

| נושא | הכלל |
|---|---|
| סטטוס הזמנה | הזמנות שסופקו בלבד, להכנסות, לביקורות ולמשך הפעילות |
| בחירת ביקורת | הראשונה לפי תאריך, שבירת שוויון לפי שעת המענה |
| חיבור ביקורות | תמיד לצמצם לשורה אחת לכל הזמנה לפני החיבור |
| שיוך ביקורת לחלון | לפי מועד הרכישה של ההזמנה, ולא לפי מועד יצירת הביקורת |
| שורות חוזרות בטבלת הפריטים | **אין להסיר אותן.** 7,088 צירופי הזמנה ומוצר חוזרים, והם כמות ולא כפילות |
| 462 המוכרים שלא מכרו | הפעלה וערך מדווחים כשתי תוצאות נפרדות |
| הגדרת הפעלה | הזמנה ראשונה בתוך 90 ימים קלנדריים מהחתימה, **בכל סטטוס**. זהו החריג היחיד לסינון שסופק |
| סוף נתוני ההזמנות | 29 באוגוסט 2018, היום האחרון שבו נרשמו לפחות עשר הזמנות. אחריו מופיעות רק הזמנות בודדות, כמעט כולן מבוטלות |
| כשירות עסקה לחלון | נכללות רק עסקאות עם 90 ימי חשיפה ומעלה עד סוף הנתונים, ובסך הכול 665 מתוך 842 |
| גבול החלון | יום קלנדרי, והיום התשעים נכלל בפנים |
| רכיב ההכנסות | לוגריתם בסיס עשר |
| רכיב הביקורות | כיווץ בייסיאני עם `k=5` |
| יעד הכיווץ | ממוצע הביקורות שבתוך החלון, ולא ממוצע של ממוצעי מוכרים |
| רכיב משך הפעילות | מנורמל, ללא הגבלת חלון. המכנה מוגדר בשורה הבאה ואינו זהה לימי החשיפה של העסקה |
| מכנה משך הפעילות | הימים מההזמנה הראשונה של המוכר ועד סוף נתוני ההזמנות. **אין להשתמש כאן בימים שמאז החתימה**, שהם מדד אחר ששולט רק בכשירות העסקה |
| אוכלוסיית הציון | מוכרים מעסקאות כשירות בלבד שיש להם הזמנה שסופקה, ובסך הכול 341 מוכרים |
| היקף מדידת הציון | הכנסות ושביעות רצון נמדדות בחלון 90 יום בלבד, משך הפעילות ללא הגבלה |
| מוכרים עם משך אפס | 69 מוכרים מקבלים ציון מינימלי במשך הפעילות, מהם 68 עם הזמנה אחת בלבד |
| מוכרים ללא פעילות בחלון | 52 מוכרים בלי הכנסה ובלי ביקורת בחלון. מקבלים רצפה בהכנסות וממוצע כללי בשביעות רצון, ומצוינים בכל הצגה |
| משקלות הרכיבים | שווים, עם טבלת רגישות לצד התוצאה |
| ערוצים קטנים | שורה נפרדת מ-20 עסקאות ומעלה, איחוד השאר, למעט `other_publicities` שמוצג כהערת שוליים |
| משפחת ההשוואות | עשרת הזוגות שבין חמשת ערוצי השיווק. לצידה מדווחות תמיד משפחות של 15 ו-21 זוגות |
| הקטגוריה הלא ידועה | מוצגת בטבלאות, מוחרגת מכל דירוג, מהמבחנים הראשיים ומכל המלצה, ומדווחת לצידם כבדיקת רגישות |
| בסיס הסימולציה | שיעור הפעלה, בהסטה של חמישית מלידי החיפוש האורגני לממומן, במוכרים ולא בכסף |

**המבחנים עצמם אינם חלק מהסקירה.** הם מחושבים במחברת 03, והתוצאות מוצגות בספר.

</div>